# N02 - Batch B smoke test
Ten-row end-to-end check on ONE GPU before the real pilot: two tiny training runs (SFT and DPO) plus a
four-prompt evaluation. Takes a few minutes and writes nothing that a reported run uses.

On ARCC2 run it as a job instead:
`srun -p gpu-h200 -A proj-606 -q proj-606 --gres=gpu:1 -t 00:40:00 --pty bash -c 'source ~/.heirloom_env; python scripts/p0.py smoke'`

In [ ]:
import os, sys
from pathlib import Path
REPO = Path(os.environ.get("HEIRLOOM_REPO", Path.cwd().parents[1]))
sys.path.insert(0, str(REPO)); os.chdir(REPO)
import pandas as pd
AGG = REPO / "results" / "P0" / "aggregate"
print("repo:", REPO)

In [ ]:
from heirloom.train import common
from heirloom.utils import paths
print("location:", paths.exec_location(), "| runtime:", common.runtime_profile())
print("models dir:", paths.models_dir())
import torch; print("gpu:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE")

In [ ]:
from heirloom.train.run import train_branch
for b in ("sft", "dpo"):
    print(b, train_branch(b, 0, max_rows=10, max_steps=2, out_root=paths.models_dir() / "SMOKE" / b))

In [ ]:
from heirloom.eval.evaluate import evaluate_model
r = evaluate_model("base", limit=4)
r["pooled_attack"]["primary"]

If both cells finish without error, Batch B works on this machine. Then submit the pilot: `bash scripts/cluster/p0_submit.sh`